# Iteration-Level Experiment Analysis

### View an experiment's progress across training iterations

**What this notebook does**

This notebook walks through iteration-level analysis of a generative-model training run (and, optionally, comparisons across several runs). It covers:

1. SAScore distributions across iterations (single experiment)
2. A helper to load `evaluation.csv` metrics from a model registry
3. Plotting validation/quality metrics (validity, novelty, QED, etc.) across iterations
4. Comparing SAScore and metric trends across **multiple** experiments

**How to use it**
- Fill in the path variables in the *Configuration* cells below (Section 1 and Section 1b).
- Run cells top to bottom. Sections that only need a single experiment (Section 2 onward through Section 5) use `base_dir`. Sections under *Multi-experiment comparison* use `base_dirs`.
- Every loader skips missing files gracefully and reports what was missing, so it's safe to run against partial data.

**Expected data layout**
```
<base_dir>/
  iter_0/relaxed/sascore.txt
  iter_1/relaxed/sascore.txt
  ...
  diffusion_wdir/model_registry/model_iter_0/run0/evaluation.csv
  diffusion_wdir/model_registry/model_iter_1/run0/evaluation.csv
  ...
```

**Notebook map**

| Section | Needs | Notes |
|---|---|---|
| 2. SAScore distributions | `base_dir` | reads `iter_*/relaxed/sascore.txt` |
| (alt view) one subplot per iteration | `base_dir` | same data as Section 2 |
| 3. Evaluation CSV helper | - | defines `load_iteration_data_from_paths`, used by Sections 5 & 6 |
| 4. Metrics plot helper | - | defines `plot_metrics_subplots`, used by Sections 5 & 6 |
| 5. Registry evaluation metrics | `base_dir` | single experiment |
| 6. Detailed metric collection | `base_dirs` | multiple experiments, uses helpers from 3 & 4 |
| Multi experiments comparison | `base_dirs` | reads `iter_*/score_filtered/sascore.txt` |
| Compare current vs. previous | `base_dirs` | reads `iter_*/relaxed/sascore.txt` |

The last two sections share one `collect_sascore_data(...)` helper (defined just above the first of them) and only differ in which sub-folder they read from.

## Section 1: Configure paths (single experiment)
Set `base_dir` to the experiment root containing `iter_*` folders.

In [ ]:
# Root directory of the experiment you want to inspect.
# Expected to contain "iter_0", "iter_1", ... folders and a "diffusion_wdir/model_registry" folder.
base_dir = "/path/to/your/base/directory"  # <-- SET THIS


## Section 1b: Configure paths (multiple experiments)
Only needed for the *Multi-experiment comparison* sections near the end of this notebook. Add one entry per experiment you want to compare; the key is the label used in plots/legends.

In [ ]:
# Map of {label shown in plots: path to experiment root}.
# Only used by the "Multi experiments comparison" sections below.
base_dirs = {
    "experiment 1": "/path/to/your/data_1",  # <-- SET THIS
    "experiment 2": "/path/to/your/data_2",  # <-- SET THIS
}

# Metrics to plot from evaluation.csv (used in Sections 5 and 6).
metrics = ["mol_stable", "atm_stable", "validity", "sanitize_validity",
           "novelty", "kl_score", "QED", "Diversity"]


#### Required imports
Import plotting and file utilities used in the analysis sections below.

In [ ]:
import os
import re
import math
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.ticker import FormatStrFormatter, MaxNLocator, MultipleLocator


## Section 2: Plot SAScore distributions across iterations
Load each `sascore.txt` file and overlay KDE curves for a visual trend comparison.

In [ ]:
plt.figure(figsize=(10, 6))

missing_files = []

# Loop over all iteration folders
for iter_num in sorted(
    [d for d in os.listdir(base_dir) if re.match(r"iter_\d+", d)],
    key=lambda x: int(re.search(r"\d+", x).group())
):
    sas_file = os.path.join(base_dir, iter_num, "relaxed", "sascore.txt")

    if os.path.exists(sas_file):
        data = pd.read_csv(sas_file, sep=r"\s+", header=None)
        sas = data[1]  # Second column
        sns.kdeplot(sas, label=f"{iter_num}", fill=True)
    else:
        missing_files.append(sas_file)

plt.xticks(range(0, 11, 1))
plt.xlabel("SAScore")
plt.ylabel("Density")
plt.title("SAScore Distribution Across Iterations")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

# Report missing files if any
if missing_files:
    print("Missing files:")
    for f in missing_files:
        print(f" - {f}")


#### Alternative view: one subplot per iteration
Plot each iteration separately to compare distributions without overlay noise.

In [ ]:
# Collect data
sas_data = []
iterations = []

for iter_num in sorted(
    [d for d in os.listdir(base_dir) if re.match(r"iter_\d+", d)],
    key=lambda x: int(re.search(r"\d+", x).group())
):
    sas_file = os.path.join(base_dir, iter_num, "relaxed", "sascore.txt")
    if os.path.exists(sas_file):
        data = pd.read_csv(sas_file, sep=r"\s+", header=None)
        sas_data.append(data[1])
        iterations.append(iter_num)

if not sas_data:
    print("No sascore.txt files found under", base_dir)
else:
    fig, axes = plt.subplots(len(sas_data), 1, figsize=(8, 2 * len(sas_data)), sharex=True)
    axes = axes if hasattr(axes, "__len__") else [axes]

    for ax, sas, label in zip(axes, sas_data, iterations):
        sns.kdeplot(sas, fill=True, ax=ax)
        ax.set_title(label)
        ax.grid(True)

    plt.tight_layout()
    plt.show()


## Section 3: Load evaluation CSV helper
Define a helper to parse evaluation CSV output into a combined DataFrame.

In [ ]:
def load_iteration_data_from_paths(file_paths, dataset_label=None):
    """
    Parse a list of evaluation.csv files (one per iteration) into a single DataFrame.

    Each file is expected to contain a header line starting with ",mol_stable" and a
    single data line. An "iteration" column (position in file_paths) is added, plus an
    optional "Dataset" label when comparing multiple experiments.
    """
    all_dfs = []
    for i, path in enumerate(file_paths):
        with open(path, "r") as f:
            lines = [line.strip() for line in f if line.strip()]

        data_lines = [line for line in lines if not line.startswith(",mol_stable")]
        if not data_lines:
            continue

        header_line = [line for line in lines if line.startswith(",mol_stable")][0]
        header = header_line.lstrip(",").split(",")
        values = data_lines[0].lstrip(",").split(",")[1:]

        df = pd.DataFrame([values], columns=header)
        df["iteration"] = i
        df["filename"] = path
        if dataset_label is not None:
            df["Dataset"] = dataset_label
        all_dfs.append(df)

    if not all_dfs:
        return pd.DataFrame()

    combined_df = pd.concat(all_dfs, ignore_index=True)

    # Convert all columns (except filename/Dataset) to numeric where possible
    for col in combined_df.columns:
        if col not in ("filename", "Dataset"):
            converted = pd.to_numeric(combined_df[col], errors="coerce")
            if converted.notna().any():
                combined_df[col] = converted

    return combined_df


## Section 4: Plot metrics helper
Define a reusable plotting function for validation and quality metrics across iterations.

In [ ]:
def plot_metrics_subplots(df, metrics, hue_col=None):
    """
    Plot multiple metrics from a DataFrame, one subplot per metric.

    Args:
        df (pd.DataFrame): data containing an "iteration" column and the metric columns.
        metrics (list[str]): column names to plot.
        hue_col (str, optional): column to use as a hue/legend (e.g. "Dataset") when
            comparing multiple experiments on the same axes.
    """
    if "iteration" not in df.columns:
        print("Column 'iteration' not found in data.")
        return

    df = df.copy()
    df["iteration"] = pd.to_numeric(df["iteration"], errors="coerce")

    existing_metrics = [m for m in metrics if m in df.columns]
    missing_metrics = [m for m in metrics if m not in df.columns]
    if missing_metrics:
        print(f"Metrics not found in data and will be skipped: {', '.join(missing_metrics)}")
    if not existing_metrics:
        print("No valid metrics found to plot.")
        return

    for metric in existing_metrics:
        df[metric] = pd.to_numeric(df[metric], errors="coerce")

    df_sorted = df.sort_values("iteration")
    num_metrics = len(existing_metrics)
    num_cols = int(math.ceil(math.sqrt(num_metrics)))
    num_rows = int(math.ceil(num_metrics / num_cols))

    fig, axes = plt.subplots(num_rows, num_cols, figsize=(5 * num_cols, 4 * num_rows), sharex=True)
    axes = axes.flatten() if hasattr(axes, "flatten") else [axes]

    palette = sns.color_palette("tab10", len(df[hue_col].unique()) if hue_col and hue_col in df.columns else len(existing_metrics))
    hue_order = sorted(df[hue_col].unique()) if hue_col and hue_col in df.columns else None

    for i, metric in enumerate(existing_metrics):
        ax = axes[i]
        if hue_col and hue_col in df.columns:
            sns.lineplot(data=df_sorted, x="iteration", y=metric, hue=hue_col,
                         hue_order=hue_order, marker="o", palette=palette,
                         linewidth=2, ax=ax, sort=False)
            if ax.legend_ is not None:
                ax.legend_.remove()
        else:
            ax.plot(df_sorted["iteration"], df_sorted[metric], marker="o",
                    color=palette[i], linewidth=2)

        ax.set_title(metric, fontsize=14, fontweight="bold")
        ax.set_xlabel("Iteration", fontsize=10)
        ax.set_ylabel(metric, fontsize=10)
        ax.grid(True, linestyle="--", alpha=0.6)
        ax.yaxis.set_major_locator(MaxNLocator(nbins=5))

    if hue_col:
        handles, labels = axes[0].get_legend_handles_labels()
        if handles:
            fig.legend(handles, labels, title=hue_col, loc="lower right")

    for j in range(num_metrics, len(axes)):
        axes[j].set_visible(False)

    fig.supxlabel("Iteration", fontsize=12)
    fig.suptitle("Metrics over Iterations", fontsize=16)
    plt.tight_layout(rect=[0, 0.03, 1, 0.95])
    plt.show()


## Section 5: Plot registry evaluation metrics
Load evaluation CSV files from the model registry (single experiment, `base_dir`) and plot metrics across iterations.

In [ ]:
wdirectory = os.path.join(base_dir, "diffusion_wdir", "model_registry")
file_paths = []
missing_files = []

for folder in sorted(
    [d for d in os.listdir(wdirectory) if re.match(r"model_iter_\d+", d)],
    key=lambda x: int(re.search(r"\d+", x).group())
):
    file_path = os.path.join(wdirectory, folder, "run0", "evaluation.csv")
    if os.path.exists(file_path):
        file_paths.append(file_path)
    else:
        missing_files.append(file_path)

df = load_iteration_data_from_paths(file_paths)
plot_metrics_subplots(df, metrics)

if missing_files:
    print("Missing files:")
    for f in missing_files:
        print(f" - {f}")


## Section 6: Detailed evaluation metric collection (multiple experiments)
Collect and plot iteration metrics from the experiment directories listed in `base_dirs` (Section 1b).

In [ ]:
def collect_iteration_files(exp_base_dir):
    """Collect evaluation.csv files for a given experiment base directory."""
    wdir = os.path.join(exp_base_dir, "diffusion_wdir", "model_registry")
    file_paths = []
    missing = []

    for folder in sorted(
        [d for d in os.listdir(wdir) if re.match(r"model_iter_\d+", d)],
        key=lambda x: int(re.search(r"\d+", x).group())
    ):
        file_path = os.path.join(wdir, folder, "run0", "evaluation.csv")
        if os.path.exists(file_path):
            file_paths.append(file_path)
        else:
            missing.append(file_path)

    return file_paths, missing


def load_iteration_data_with_number(file_paths, dataset_label):
    """Load CSV files into one DataFrame, extracting the iteration number from the filename."""
    dfs = []
    for path in file_paths:
        try:
            df = pd.read_csv(path)
            iteration_match = re.search(r"model_iter_(\d+)", path)
            df["iteration"] = int(iteration_match.group(1)) if iteration_match else None
            df["Dataset"] = dataset_label
            dfs.append(df)
        except Exception as e:
            print(f"Error reading {path}: {e}")
    return pd.concat(dfs, ignore_index=True) if dfs else pd.DataFrame()


# ---------- MAIN ----------
all_dfs = []
all_missing = []

for label, exp_dir in base_dirs.items():
    files, missing = collect_iteration_files(exp_dir)
    df = load_iteration_data_with_number(files, label)
    if not df.empty:
        all_dfs.append(df)
    all_missing.extend(missing)

if all_dfs:
    df_all = pd.concat(all_dfs, ignore_index=True)
    plot_metrics_subplots(df_all, metrics, hue_col="Dataset")
else:
    print("No data loaded.")

if all_missing:
    print("Missing files:")
    for f in all_missing:
        print(f" - {f}")


### Multi experiments comparison
Both remaining sections collect SAScore values across `base_dirs` and iterations — they only differ in which sub-folder they read from (`score_filtered` vs. `relaxed`) and how they plot the result, so the collection step is factored into one `collect_sascore_data(...)` helper below.

In [ ]:
def collect_sascore_data(base_dirs, subfolder):
    """
    Collect SAScore values across all iterations of every experiment in base_dirs.

    Parameters
    ----------
    base_dirs : dict[str, str]
        {label: experiment root path}, as configured in Section 1b.
    subfolder : str
        Sub-folder under each iter_N directory to read sascore.txt from
        (e.g. "score_filtered" or "relaxed").

    Returns
    -------
    pd.DataFrame with columns: SAScore, Iteration, Dataset
    """
    all_dfs = []
    for label, exp_dir in base_dirs.items():
        iter_dirs = sorted(
            [d for d in os.listdir(exp_dir) if re.match(r"iter_\d+", d)],
            key=lambda x: int(re.search(r"\d+", x).group())
        )
        for iter_dir in iter_dirs:
            sas_file = os.path.join(exp_dir, iter_dir, subfolder, "sascore.txt")
            if os.path.exists(sas_file):
                try:
                    data = pd.read_csv(sas_file, sep=r"\s+", header=None)
                    df = pd.DataFrame({
                        "SAScore": data[1],
                        "Iteration": int(re.search(r"\d+", iter_dir).group()),
                        "Dataset": label,
                    })
                    all_dfs.append(df)
                except Exception as e:
                    print(f"Error reading {sas_file}: {e}")

    return pd.concat(all_dfs, ignore_index=True) if all_dfs else pd.DataFrame()


Compare SAScore distributions across the experiments listed in `base_dirs`, one subplot per iteration.

In [ ]:
df_all = collect_sascore_data(base_dirs, subfolder="score_filtered")

if df_all.empty:
    print("No data found.")
else:
    iterations = sorted(df_all["Iteration"].unique())
    num_iters = len(iterations)

    fig, axes = plt.subplots(num_iters, 1, figsize=(8, 2 * num_iters), sharex=True)
    axes = axes if hasattr(axes, "__len__") else [axes]

    for ax, iter_num in zip(axes, iterations):
        df_iter = df_all[df_all["Iteration"] == iter_num]
        sns.kdeplot(data=df_iter, x="SAScore", hue="Dataset", fill=True, ax=ax)
        ax.set_title(f"Iteration {iter_num}")
        ax.grid(True)

    plt.xlabel("SAScore")
    plt.tight_layout()
    plt.show()


#### Compare current and previous iteration distributions
Overlay current and previous iteration SAScore curves for each dataset in `base_dirs` (this section reads from the `relaxed` sub-folder instead of `score_filtered`).

In [ ]:
df_all = collect_sascore_data(base_dirs, subfolder="relaxed")

if df_all.empty:
    print("No data found.")
else:
    df_all = pd.concat(all_dfs, ignore_index=True)

    datasets = df_all["Dataset"].unique()
    iterations = sorted(df_all["Iteration"].unique())
    num_iters = len(iterations)
    palette = dict(zip(datasets, sns.color_palette("tab10", len(datasets))))

    fig, axes = plt.subplots(num_iters, 1, figsize=(8, 2 * num_iters), sharex=True)
    axes = axes if hasattr(axes, "__len__") else [axes]

    for ax, iter_num in zip(axes, iterations):
        for dataset in datasets:
            df_dataset = df_all[df_all["Dataset"] == dataset]

            df_curr = df_dataset[df_dataset["Iteration"] == iter_num]
            if not df_curr.empty:
                sns.kdeplot(df_curr["SAScore"], fill=True, ax=ax, alpha=0.3,
                            label=f"{dataset} current", color=palette[dataset])

            prev_iter = iter_num - 1
            df_prev = df_dataset[df_dataset["Iteration"] == prev_iter]
            if not df_prev.empty:
                sns.kdeplot(df_prev["SAScore"], fill=False, ax=ax, linestyle="--",
                            label=f"{dataset} prev", color=palette[dataset])

        ax.tick_params(axis="x", labelbottom=True)
        ax.xaxis.set_major_locator(MultipleLocator(1))
        ax.set_title(f"Iteration {iter_num}")
        ax.grid(True)
        ax.legend(fontsize=7, loc="upper right")

    plt.xlabel("SAScore")
    plt.tight_layout()
    plt.show()
